# 1 · Caliby / SolubleCaliby - sequence design
Structure-conditioned design with the Caliby package (the reference pipeline).
Reads the shared 25-protein bundle; writes `designs_<model>.csv` in the common schema.
**Runtime → Change runtime type → GPU** before running.

In [ ]:
#@title Step 0 - Upload & unzip the design bundle
#@markdown Upload **design_bundle.zip** (contains `design_common.py`,
#@markdown `design_input_proteins.csv`, and `structures/`).
#@markdown Build it locally with `design/make_bundle.sh`.
import os, zipfile
from google.colab import files

if not os.path.exists("design_common.py"):
    print("Upload design_bundle.zip:")
    up = files.upload()
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall(".")
    # if it unzipped into a 'design/' subdir, hoist contents to CWD
    if os.path.exists("design/design_common.py") and not os.path.exists("design_common.py"):
        import shutil
        for item in os.listdir("design"):
            shutil.move(os.path.join("design", item), item)
print("Bundle ready:", sorted(os.listdir(".")))

In [ ]:
#@title Install Caliby
import os, subprocess, sys
if not os.path.isfile("CALIBY_READY"):
    subprocess.run([sys.executable,"-m","pip","install",
        "caliby[af2] @ git+https://github.com/ProteinDesignLab/caliby.git",
        "--quiet"], check=True)
    open("CALIBY_READY","w").write("done")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set GPU runtime!)")

In [ ]:
#@title Step 1 - Import shared config and show the LOCKED settings
import design_common as dc
proteins = dc.load_inputs()           # 25 templates, structure paths resolved
print(f"Loaded {len(proteins)} design templates")
print("\n=== LOCKED CONFIG (identical across all model notebooks) ===")
import dataclasses, json
cfg = {k: v for k, v in dataclasses.asdict(dc.CONFIG).items() if k != "deviations"}
print(json.dumps(cfg, indent=2, default=str))
display(proteins[["uniprot_id","species","domain","rank_class","sequence_length"]])

In [ ]:
#@title Choose variant
#@markdown `caliby` = standard; `soluble_caliby` = transmembrane-excluded (SolubleMPNN analogue)
variant = "caliby"  #@param ["caliby", "soluble_caliby", "soluble_caliby_v1"]
MODEL = {"caliby":"Caliby","soluble_caliby":"SolubleCaliby","soluble_caliby_v1":"SolubleCaliby_v1"}[variant]
SOLUBLE = variant != "caliby"
from caliby import load_model
model = load_model(variant)
num_workers = 2
print("Loaded", MODEL)

## Comparability notes - Caliby

These are the points where Caliby touches the locked settings. Anything that
**deviates** is recorded via `dc.CONFIG.note_deviation(...)` so it lands in the
output manifest.


- **Temperature, num_seqs, omit_aas**: honoured exactly from `dc.CONFIG`.
- **Sampler**: Caliby samples temperature-only (no top-k/top-p) - already matches the locked scheme.
- **Self-consistency**: we do **not** use Caliby's built-in `self_consistency_eval` here.
  Self-consistency is run once for every model in notebook 5, so the refold protocol is identical.

In [ ]:
#@title Step 2 - Smoke test: design ONE sequence for the shortest protein
import pandas as pd
_p = proteins.sort_values("sequence_length").iloc[0]
_res = model.sample(pdb_paths=[_p.structure_path], num_seqs_per_pdb=1,
                    batch_size=1, temperature=dc.CONFIG.temperature,
                    omit_aas=list(dc.CONFIG.omit_aas) or None,
                    num_workers=num_workers, out_dir="outputs/_smoke")
_seq = pd.DataFrame(_res).iloc[0]["seq"]
print(f"{_p.uniprot_id} ({_p.species}) len={_p.sequence_length}")
print("WT  :", _p.wt_sequence[:60])
print("DES :", _seq[:60])
assert len(_seq) == _p.sequence_length, "length mismatch!"
assert set(_seq) <= set(dc.CANONICAL_AA), "non-canonical residue!"
print("✓ smoke test OK")

In [ ]:
#@title Step 3 - Design all 25 proteins  (num_seqs at locked temperature)
import pandas as pd
from tqdm.auto import tqdm
rows = []
for p in tqdm(list(proteins.itertuples()), desc="Caliby design"):
    res = model.sample(pdb_paths=[p.structure_path],
                       num_seqs_per_pdb=dc.CONFIG.num_seqs_per_protein,
                       batch_size=1, temperature=dc.CONFIG.temperature,
                       omit_aas=list(dc.CONFIG.omit_aas) or None,
                       num_workers=num_workers,
                       out_dir=f"outputs/seq_des/{p.uniprot_id}")
    rdf = pd.DataFrame(res).reset_index(drop=True)
    for i, rr in rdf.iterrows():
        rows.append(dc.make_record(p, model=MODEL, sample_idx=i,
                                   designed_sequence=rr["seq"],
                                   model_score=float(rr["U"]), score_type="caliby_U",
                                   soluble_variant=SOLUBLE))
print(f"Generated {len(rows)} sequences")

In [ ]:
#@title Step 4 - Validate (faithful + comparable) and write outputs
df = dc.finalize(rows, model=MODEL, strict=True)   # raises if a guard fails
dc.write_designs(df, MODEL)
dc.write_fasta(df, MODEL)

# Quick faithfulness readout: per-protein WT sequence recovery distribution
import numpy as np
rec = df.apply(lambda r: sum(a==b for a,b in zip(r.designed_sequence, r.wt_sequence))/r.seq_length, axis=1)
print(f"\nSeq-recovery vs WT - median {rec.median():.1%}, "
      f"IQR [{rec.quantile(.25):.1%}, {rec.quantile(.75):.1%}]")
print("(Inverse-folding designs typically recover ~30-55% of WT; "
      "near-100% means the sampler is too cold / stuck, near-5% means random.)")

from google.colab import files
files.download(str(dc.OUTPUT_DIR / f"designs_{MODEL.replace('/','_').replace('-','-')}.csv"))